Use GridSearchCV for hyperparameter tuning of the best model by manually implementing an exhaustive search algorithm, evaluating multiple hyperparameter combinations without using built-in grid search methods.

In [1]:
import pandas as pd
import numpy as np
from itertools import product
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("heart_disease_train.csv")

features = ["age", "trestbps", "chol", "thalach", "oldpeak"]
target = "target"

df = df[features + [target]].copy()

for col in features + [target]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna()

X = df[features].to_numpy(dtype=float)
y = df[target].to_numpy(dtype=int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.25,
    random_state=42, stratify=y_train
)

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))

Training: 47
Validation: 16
Testing: 16


In [8]:
def make_model(name, params=None):
    params = params or {}

    if name == "Logistic Regression":
        return make_pipeline(
            StandardScaler(),
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                **params
            )
        )

    elif name == "Decision Tree":
        return DecisionTreeClassifier(
            class_weight="balanced",
            random_state=42,
            **params
        )

    elif name == "Random Forest":
        return RandomForestClassifier(
            n_estimators=100,
            class_weight="balanced",
            random_state=42,
            **params
        )

    elif name == "SVM":
        return make_pipeline(
            StandardScaler(),
            SVC(
                class_weight="balanced",
                probability=True,
                random_state=42,
                **params
            )
        )

    elif name == "KNN":
        return make_pipeline(
            StandardScaler(),
            KNeighborsClassifier(**params)
        )

    raise ValueError(f"Unknown model: {name}")

In [9]:
param_grids = {
    "Logistic Regression": {
        "C": [0.1, 1, 10],
        "solver": ["liblinear", "lbfgs"]
    },
    "Decision Tree": {
        "max_depth": [3, 5, 10, None],
        "min_samples_leaf": [1, 3, 5]
    },
    "Random Forest": {
        "max_depth": [5, 10, None],
        "min_samples_leaf": [1, 3, 5],
        "max_features": ["sqrt", None]
    },
    "SVM": {
        "C": [0.1, 1, 10],
        "kernel": ["linear", "rbf"]
    },
    "KNN": {
        "n_neighbors": [3, 5, 7, 9],
        "weights": ["uniform", "distance"]
    }
}

param_grid = param_grids[best_model_name]
param_names = list(param_grid.keys())
param_combinations = list(product(*param_grid.values()))

print("Selected model:", best_model_name)
print("Total combinations:", len(param_combinations))

Selected model: Logistic Regression
Total combinations: 6


In [10]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search_results = []

for combination in param_combinations:
    params = dict(zip(param_names, combination))
    fold_f1_scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        model = make_model(best_model_name, params)

        model.fit(X_train[train_idx], y_train[train_idx])
        pred = model.predict(X_train[val_idx])

        precision, recall, f1 = calculate_metrics(
            y_train[val_idx], pred
        )
        fold_f1_scores.append(f1)

    search_results.append({
        "Parameters": params,
        "Mean F1 Score": np.mean(fold_f1_scores),
        "Std F1 Score": np.std(fold_f1_scores)
    })

search_df = pd.DataFrame(search_results).sort_values(
    "Mean F1 Score", ascending=False
)

print(search_df.head(10).to_string(index=False))

                       Parameters  Mean F1 Score  Std F1 Score
  {'C': 1, 'solver': 'liblinear'}       0.818223      0.090667
      {'C': 1, 'solver': 'lbfgs'}       0.805037      0.078426
 {'C': 10, 'solver': 'liblinear'}       0.805037      0.078426
     {'C': 10, 'solver': 'lbfgs'}       0.805037      0.078426
    {'C': 0.1, 'solver': 'lbfgs'}       0.797711      0.109149
{'C': 0.1, 'solver': 'liblinear'}       0.758608      0.080389


In [11]:
best_params = search_df.iloc[0]["Parameters"]

tuned_model = make_model(best_model_name, best_params)
tuned_model.fit(
    np.vstack([X_train, X_val]),
    np.concatenate([y_train, y_val])
)

print("Best model:", best_model_name)
print("Best parameters:", best_params)
print("Best cross-validation F1:", round(
    search_df.iloc[0]["Mean F1 Score"], 4
))

Best model: Logistic Regression
Best parameters: {'C': 1, 'solver': 'liblinear'}
Best cross-validation F1: 0.8182


In [12]:
test_pred = tuned_model.predict(X_test)

precision, recall, f1 = calculate_metrics(y_test, test_pred)
accuracy = np.mean(test_pred == y_test)

print("Final Tuned Model Results")
print("Model:", best_model_name)
print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))

Final Tuned Model Results
Model: Logistic Regression
Accuracy: 0.6875
Precision: 0.7692
Recall: 0.8333
F1 Score: 0.8
